In [19]:
import numpy as np

# Each message: [number of scam words, has a link (1/0)]
X = np.array([[0, 0], [1, 0], [0, 1], [3, 1], [4, 1], [5, 0]], dtype=float)
y = np.array([0, 0, 0, 1, 1, 1], dtype=float)   # 1 = scam

weights = np.zeros(2)
bias = 0.0
learning_rate = 0.1

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for step in range(201):
    # 1. Guess
    p = sigmoid(X @ weights + bias)

    # 2. Measure the mistake (loss)
    loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))

    # 3. Adjust (gradient descent)
    error = p - y
    weights -= learning_rate * (X.T @ error) / len(y)
    bias -= learning_rate * error.mean()

    if step % 50 == 0:
        print(f"step {step}: loss {loss:.3f}, weights {weights.round(2)}")

step 0: loss 0.693, weights [0.09 0.01]
step 50: loss 0.288, weights [ 0.86 -0.06]
step 100: loss 0.200, weights [ 1.11 -0.09]
step 150: loss 0.153, weights [ 1.3  -0.07]
step 200: loss 0.125, weights [ 1.44 -0.04]


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("../data/SMSSpamCollection", sep="\t", names=["label", "message"])
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [23]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

nn = MLPClassifier(hidden_layer_sizes=(128,64), max_iter=30, random_state=42)
nn.fit(X_train_tfidf, y_train)
pred_nn = nn.predict(X_test_tfidf)

print(classification_report(y_test, pred_nn))
print(confusion_matrix(y_test, pred_nn, labels=["spam", "ham"]))

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.98      0.88      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

[[131  18]
 [  3 963]]
